In [1]:
!pip install groq python-dotenv

In [2]:
# setup

import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv() 

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Say hello in one short sentence."}]
)

print(response.choices[0].message.content)

Hello!


In [3]:
from Clinical_Ontology import CHIEF_COMPLAINT_ONTOLOGY, STANDARD_HISTORY_SECTIONS

In [4]:
# conversation state

def detect_chief_complaint(patient_text):
    
    known_complaints = list(CHIEF_COMPLAINT_ONTOLOGY.keys())
    
    prompt = f"""The patient said: "{patient_text}"
    Which of these known complaints does this best match: "{known_complaints}"?
    Respond only with the matching category name from the list, or "unknown" if none fit. No Explaination, just the category name"""

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages = [{"role": "user", "content": prompt}],
        temperature = 0
    )
    return response.choices[0].message.content.strip()

In [5]:
test_result = detect_chief_complaint("I've had really bad chest pain since morning")
print(test_result)

chest_pain


In [6]:
conversation_state = {
    "cheif_complaint": None,
    "current_section": "chief_complaint",
    "question_index": 0,
    "transcript": []
}

In [7]:
def get_next_question(state):
    
    if state["current_section"] == "chief_complaint":
        questions = CHIEF_COMPLAINT_ONTOLOGY[state["chief_complaint"]]["follow_up_questions"]
        
        if state["question_index"] < len(questions):
            return questions[state["question_index"]]
        else:
            state["current_section"] = "past_medical_surgical_history"
            state["question_index"] = 0
            return get_next_question(state)
    
    elif state["current_section"] in STANDARD_HISTORY_SECTIONS:
        questions = STANDARD_HISTORY_SECTIONS[state["current_section"]]
        
        if state["question_index"] < len(questions):
            return questions[state["question_index"]]
        else:
            section_order = list(STANDARD_HISTORY_SECTIONS.keys())
            current_idx = section_order.index(state["current_section"])
            
            if current_idx + 1 < len(section_order):
                state["current_section"] = section_order[current_idx + 1]
                state["question_index"] = 0
                return get_next_question(state)
            else:
                state["current_section"] = "done"
                return None
    
    return None

In [8]:
conversation_state["chief_complaint"] = "chest_pain"

q1 = get_next_question(conversation_state)
print(q1)

Where exactly is the pain located?


In [19]:
# the conversation loop

def run_history_intake(state):
    
    while state["current_section"] != "done":
        question = get_next_question(state)
        
        if question is None:
            break
        
        print(f"\nAI: {question}")
        answer = input("Patient: ")
        
        state["transcript"].append({
            "question": question,
            "answer": answer,
            "section": state["current_section"]
        })

        if state["current_section"] == "chief_complaint":
            flagged, flags = check_red_flag(state["chief_complaint"], answer)
            if flagged:
                state["flagged_red_flags"] = state.get("flagged_red_flags", [])
                state["flagged_red_flags"].append({"answer": answer, "matched_flags":flags})
                print(" ALERT : RED FLAG DETECTED - PATIENT WILL IMMEDIATELY NEED ASSISTANCE")
        
        state["question_index"] += 1
    
    print("\n✅ History intake complete.")

    if state.get("flagged_red_flags"):
        print(f"\n {len(state['flagged_red_flags'])}, red flags were detected")
    return state

In [25]:
conversation_state = {
    "chief_complaint": "chest_pain",
    "current_section": "chief_complaint",
    "question_index": 0,
    "transcript": []
}

final_state = run_history_intake(conversation_state)


AI: Where exactly is the pain located?


Patient:  in the middle of my chest



AI: When did it start — suddenly or gradually?


Patient:  suddenly



AI: How would you describe the pain — sharp, dull, burning, cramping?


...


AI: Any joint pains, swelling, or skin issues?


Patient:  



AI: Any recent changes in vision, hearing, or balance?


Patient:  



✅ History intake complete.

 1, red flags were detected


In [26]:
def check_red_flag(chief_complaint, patient_answer):
    
    if chief_complaint not in CHIEF_COMPLAINT_ONTOLOGY:
        return False, None
    
    red_flags = CHIEF_COMPLAINT_ONTOLOGY[chief_complaint]["red_flags"]
    complaint_readable = chief_complaint.replace("_", " ")
    
    prompt = f"""A patient came in with a chief complaint of: {complaint_readable}.

During follow-up questioning, the patient just answered: "{patient_answer}"

Given that this patient has {complaint_readable}, does their answer suggest any of these red flag warning signs: {red_flags}?
Respond with ONLY "yes" or "no"."""
    
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    
    result = response.choices[0].message.content.strip().lower()
    is_flagged = "yes" in result
    
    return is_flagged, red_flags if is_flagged else None

In [27]:
flagged, flags = check_red_flag("chest_pain", "yes I'm sweating a lot and feel dizzy")
print(flagged, flags)

True ['chest pain with breathlessness', 'chest pain with sweating', 'chest pain radiating to arm or jaw', 'chest pain with fainting or dizziness']


In [24]:
# def check_red_flag_debug(chief_complaint, patient_answer):
#     red_flags = CHIEF_COMPLAINT_ONTOLOGY[chief_complaint]["red_flags"]

#     prompt = f"""Patient's answer: "{patient_answer}"

#     Does this answer suggest any of the red flag warning signs: "{red_flags}"?
#     Respond only with "yes" or "no". """

#     print("PROMPT SENT:\n", prompt)
#     print("\n---\n")

#     response = client.chat.completions.create(
#         model="openai/gpt-oss-120b",
#         messages=[{"role": "user", "content": prompt}],
#         temperature=0
#     )

#     raw_result = response.choices[0].message.content
#     print("RAW MODEL OUTPUT:", repr(raw_result))

#     return raw_result

# check_red_flag_debug("chest_pain", "yes, i'm sweating a lot and feeling dizzy")